# 03 — GPU Demand

This notebook characterises GPU usage across the user base: which groups are using GPUs in production, what GPU memory they need, which frameworks they use, and the measured speedup compared to CPU-only runs.

**Run `00_setup.ipynb` first.**

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sheets_client import load_sheets, explode_semicolons, map_range_labels, GPU_MEMORY_LABELS

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

dfs = load_sheets()
gpu = dfs.get('GpuInfo', pd.DataFrame())
print(f"GpuInfo records: {len(gpu)}")
if not gpu.empty:
    print(f"Columns: {list(gpu.columns)}")

---
## Chart 1 — GPU Status Distribution

Shows how many codes are using GPUs in production, investigating GPU use, GPU-only, or unknown.

**What to look for:** What fraction of codes are already GPU-dependent? How many are investigating — these are candidates for near-term GPU demand growth.

In [ ]:
GPU_STATUS_LABELS = {
    'production': 'Production use',
    'investigating': 'Investigating / testing',
    'gpu_only': 'GPU-only (no CPU alternative)',
    'not_used': 'Not used / not applicable',
    'dont_know': "Don't know",
}

if gpu.empty or 'gpu_status' not in gpu.columns:
    print("No GPU status data available.")
else:
    status_counts = (
        gpu['gpu_status']
        .dropna()
        .str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: GPU_STATUS_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        status_counts.values,
        labels=status_counts.index,
        autopct=lambda p: f'{p:.1f}%\n({int(round(p * status_counts.sum() / 100))})',
        startangle=130,
        textprops={'fontsize': 9},
    )
    ax.set_title('GPU Status Distribution\n(per code entry)', fontsize=13, pad=16)
    plt.tight_layout()
    plt.show()
    print(status_counts.to_string())

---
## Chart 2 — GPU Memory Needed

Shows the distribution of GPU memory requirements across codes that use GPUs.

**What to look for:** Are most codes compatible with standard 16–40 GB GPU cards, or is there demand for high-memory GPUs (>40 GB or >80 GB)?

In [ ]:
GPU_MEM_ORDER = list(GPU_MEMORY_LABELS.values())

if gpu.empty or 'gpu_memory_needed' not in gpu.columns:
    print("No GPU memory data available.")
else:
    gm = (
        gpu['gpu_memory_needed']
        .dropna()
        .str.strip()
        .loc[lambda s: s != '']
    )
    gm_counts = map_range_labels(gm, GPU_MEMORY_LABELS).value_counts()
    gm_counts = gm_counts.reindex([c for c in GPU_MEM_ORDER if c in gm_counts.index])

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(gm_counts.index, gm_counts.values, color=sns.color_palette('muted')[1])
    for bar, val in zip(bars, gm_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('GPU memory required')
    ax.set_ylabel('Number of code entries')
    ax.set_title('GPU Memory Needed by Code', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Chart 3 — GPU Frameworks Used

Counts of GPU frameworks across all GpuInfo entries. Frameworks field may contain multiple values separated by semicolons.

**What to look for:** CUDA dominance vs OpenCL, ROCm, or framework-specific libraries. This informs GPU hardware procurement and driver support decisions.

In [ ]:
if gpu.empty or 'frameworks' not in gpu.columns:
    print("No GPU framework data available.")
else:
    fw = explode_semicolons(gpu, 'frameworks')
    fw_counts = fw.str.strip().value_counts()

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(fw_counts.index[::-1], fw_counts.values[::-1], color=sns.color_palette('muted')[4])
    for bar, val in zip(bars, fw_counts.values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=9)
    ax.set_xlabel('Number of code entries')
    ax.set_title('GPU Frameworks Used', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Chart 4 — NVLink / High-Bandwidth Interconnect Sensitivity

Shows whether codes require NVLink or high-bandwidth GPU-to-GPU interconnect for multi-GPU runs.

**What to look for:** NVLink-sensitive workloads require specific hardware; standard PCIe-connected GPUs may be insufficient.

In [ ]:
NVLINK_LABELS = {
    'yes': 'Yes — NVLink required',
    'no': 'No — PCIe sufficient',
    'dont_know': "Don't know",
    'not_applicable': 'Not applicable',
    'not_tested': 'Not tested',
}

if gpu.empty or 'nvlink_needed' not in gpu.columns:
    print("No NVLink data available.")
else:
    nv = (
        gpu['nvlink_needed']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: NVLINK_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(nv.index, nv.values, color=sns.color_palette('muted')[5])
    for bar, val in zip(bars, nv.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('NVLink requirement')
    ax.set_ylabel('Number of code entries')
    ax.set_title('NVLink / High-Bandwidth Interconnect Sensitivity', fontsize=13)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

---
## Chart 5 — Multi-Node GPU Requirements

Shows how many codes require multiple nodes with GPUs (vs single-node multi-GPU).

**What to look for:** Multi-node GPU jobs require specific network fabric (InfiniBand + NVLink bridges). Identifying these early helps procurement planning.

In [ ]:
MULTINODE_LABELS = {
    'yes': 'Yes — multi-node GPU',
    'no': 'No — single-node only',
    'dont_know': "Don't know",
    'not_tested': 'Not tested',
    'not_applicable': 'Not applicable',
}

if gpu.empty or 'multinode_gpu' not in gpu.columns:
    print("No multi-node GPU data available.")
else:
    mn = (
        gpu['multinode_gpu']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: MULTINODE_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(mn.index, mn.values, color=sns.color_palette('muted')[0])
    for bar, val in zip(bars, mn.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Multi-node GPU requirement')
    ax.set_ylabel('Number of code entries')
    ax.set_title('Multi-Node GPU Requirements', fontsize=13)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

---
## Chart 6 — GPU Jobs Per Year

Histogram of annual GPU job counts where provided. Only numeric entries are plotted.

**What to look for:** What is the total GPU demand? Are there codes with very high GPU job counts that will dominate GPU capacity?

In [ ]:
if gpu.empty or 'gpu_jobs_per_year' not in gpu.columns:
    print("No GPU jobs per year data available.")
else:
    gj = pd.to_numeric(gpu['gpu_jobs_per_year'], errors='coerce').dropna()
    gj = gj[gj > 0]

    if gj.empty:
        print("No numeric GPU job count values found.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(gj, bins=25, color=sns.color_palette('muted')[2], edgecolor='white', linewidth=0.5)
        ax.set_xlabel('GPU jobs per year')
        ax.set_ylabel('Number of code entries')
        ax.set_title('GPU Jobs Per Year Distribution', fontsize=13)
        plt.tight_layout()
        plt.show()
        print(f"Records: {len(gj)}  |  Min: {gj.min():.0f}  |  Median: {gj.median():.0f}  |  Max: {gj.max():.0f}")

---
## Chart 7 — CPU vs GPU Wall Time (speedup scatter)

Each point is a code with documented CPU-only and GPU wall times for equivalent workloads. Colour encodes the speedup ratio (GPU speedup = cpu_wall_time / gpu_wall_time). Only rows with both values present are plotted.

**What to look for:** High speedup ratios confirm GPU value for those workloads. Very low speedup may indicate the code does not benefit significantly from GPU acceleration.

In [ ]:
cols_needed = {'cpu_wall_time_hours', 'gpu_wall_time_hours'}

if gpu.empty or not cols_needed.issubset(gpu.columns):
    print("CPU vs GPU wall-time columns not found in GpuInfo.")
else:
    df_sp = gpu[['cpu_wall_time_hours', 'gpu_wall_time_hours']].copy()
    df_sp['cpu_wall_time_hours'] = pd.to_numeric(df_sp['cpu_wall_time_hours'], errors='coerce')
    df_sp['gpu_wall_time_hours'] = pd.to_numeric(df_sp['gpu_wall_time_hours'], errors='coerce')
    df_sp = df_sp.dropna()
    df_sp = df_sp[(df_sp['cpu_wall_time_hours'] > 0) & (df_sp['gpu_wall_time_hours'] > 0)]

    if df_sp.empty:
        print("No paired CPU/GPU wall-time records found.")
    else:
        df_sp['speedup'] = df_sp['cpu_wall_time_hours'] / df_sp['gpu_wall_time_hours']

        fig, ax = plt.subplots(figsize=(9, 6))
        sc = ax.scatter(
            df_sp['cpu_wall_time_hours'],
            df_sp['gpu_wall_time_hours'],
            c=np.log10(df_sp['speedup']),
            cmap='RdYlGn',
            s=70, alpha=0.8, edgecolors='grey', linewidths=0.4
        )
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label('log10(speedup = CPU time / GPU time)', fontsize=9)

        # Diagonal reference line (speedup = 1)
        lims = [
            min(df_sp['cpu_wall_time_hours'].min(), df_sp['gpu_wall_time_hours'].min()) * 0.8,
            max(df_sp['cpu_wall_time_hours'].max(), df_sp['gpu_wall_time_hours'].max()) * 1.2,
        ]
        ax.plot(lims, lims, 'k--', linewidth=0.8, label='Speedup = 1 (no gain)')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('CPU wall time (hours, log scale)')
        ax.set_ylabel('GPU wall time (hours, log scale)')
        ax.set_title('CPU vs GPU Wall Time\n(colour = log speedup, points above diagonal = GPU faster)', fontsize=12)
        ax.legend(fontsize=8)
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Points: {len(df_sp)}  |  Median speedup: {df_sp['speedup'].median():.1f}x  |  Max speedup: {df_sp['speedup'].max():.1f}x")